# Modal ARC-AGI-2 Volume setup

This one-time setup downloads the complete public ARC-AGI-2 `training` and
`evaluation` directories from one pinned official commit. It verifies the
archive digest, validates all 1,120 JSON tasks, and installs them durably at:

`/mnt/diffusion-arc-models/data/raw/arc-agi-2`

Attach the writable `diffusion-arc-models` Volume. A matching existing install
is verified and reused; an unknown or partial target is never overwritten.
No GPU is required.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import tarfile
import tempfile
import urllib.request

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
DATA_PARENT = VOLUME_ROOT / 'data'
DATASET_ROOT = DATA_PARENT / 'raw' / 'arc-agi-2'
MANIFEST_PATH = DATA_PARENT / 'arc-agi-2-manifest.json'

SOURCE_REPOSITORY = 'https://github.com/arcprize/ARC-AGI-2'
SOURCE_COMMIT = 'f3283f727488ad98fe575ea6a5ac981e4a188e49'
ARCHIVE_URL = (
    'https://codeload.github.com/arcprize/ARC-AGI-2/tar.gz/'
    f'{SOURCE_COMMIT}'
)
EXPECTED_ARCHIVE_SHA256 = (
    '179681bc01d55810ee0e90d2717ef04b8d130496541c4853479a8d6fffa4b975'
)
EXPECTED_COUNTS = {'training': 1000, 'evaluation': 120}
TASK_NAME_PATTERN = re.compile(r'[0-9a-f]{8}\.json')

if not VOLUME_ROOT.is_dir():
    raise RuntimeError(f'Attach the writable Modal Volume at {VOLUME_ROOT}')
DATA_PARENT.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)
    if hasattr(os, 'sync'):
        os.sync()


def validate_task_file(path: Path) -> None:
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(payload, dict) or set(payload) != {'train', 'test'}:
        raise ValueError(f'Unexpected ARC task structure: {path}')
    if not isinstance(payload['train'], list) or not isinstance(payload['test'], list):
        raise TypeError(f'ARC task splits must be arrays: {path}')


def build_file_manifest(root: Path) -> tuple[dict[str, str], dict[str, int]]:
    file_hashes: dict[str, str] = {}
    counts: dict[str, int] = {}
    for split, expected_count in EXPECTED_COUNTS.items():
        split_root = root / split
        files = tuple(sorted(split_root.glob('*.json')))
        if len(files) != expected_count:
            raise RuntimeError(
                f'{split} task count differs: {len(files)} != {expected_count}'
            )
        unexpected = [path.name for path in files if TASK_NAME_PATTERN.fullmatch(path.name) is None]
        if unexpected:
            raise RuntimeError(f'Unexpected task filenames in {split}: {unexpected[:5]}')
        for path in files:
            validate_task_file(path)
            relative = str(path.relative_to(root))
            file_hashes[relative] = sha256_file(path)
        counts[split] = len(files)
    return file_hashes, counts


def verify_installed_dataset() -> dict[str, object]:
    if not DATASET_ROOT.is_dir() or not MANIFEST_PATH.is_file():
        raise FileNotFoundError('ARC dataset or provenance manifest is missing')
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    expected_identity = {
        'repository': SOURCE_REPOSITORY,
        'commit': SOURCE_COMMIT,
        'archive_sha256': EXPECTED_ARCHIVE_SHA256,
    }
    if manifest.get('source') != expected_identity:
        raise RuntimeError('Existing ARC dataset manifest has a different source identity')
    hashes, counts = build_file_manifest(DATASET_ROOT)
    if manifest.get('counts') != counts or manifest.get('files') != hashes:
        raise RuntimeError('Existing ARC dataset differs from its durable manifest')
    return manifest


print('Target:', DATASET_ROOT)
print('Pinned source:', SOURCE_COMMIT)


## Download and atomic installation

This cell performs the only network access. It refuses to replace an existing
directory unless that directory already validates against the same manifest.


In [ ]:
if DATASET_ROOT.exists() or MANIFEST_PATH.exists():
    manifest = verify_installed_dataset()
    print('ARC-AGI-2 is already installed and verified.')
else:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    staging_root = DATA_PARENT / f'.arc-agi-2-staging-{run_id}'
    if staging_root.exists():
        raise RuntimeError(f'Refusing to reuse staging directory: {staging_root}')
    staging_root.mkdir(parents=True)

    with tempfile.TemporaryDirectory(prefix='arc-agi-2-download-') as temporary:
        archive_path = Path(temporary) / 'arc-agi-2.tar.gz'
        request = urllib.request.Request(
            ARCHIVE_URL,
            headers={'User-Agent': 'diffusion-arc-dataset-setup/1'},
        )
        with urllib.request.urlopen(request, timeout=120) as response:
            if response.geturl().split('/')[2] != 'codeload.github.com':
                raise RuntimeError(f'Unexpected archive host: {response.geturl()}')
            with archive_path.open('wb') as handle:
                while chunk := response.read(1024 * 1024):
                    handle.write(chunk)

        archive_sha256 = sha256_file(archive_path)
        if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
            raise RuntimeError(
                f'ARC archive SHA-256 differs: {archive_sha256}'
            )

        archive_prefix = f'ARC-AGI-2-{SOURCE_COMMIT}/data/'
        with tarfile.open(archive_path, mode='r:gz') as archive:
            selected: list[tuple[tarfile.TarInfo, Path]] = []
            for member in archive.getmembers():
                if not member.isfile() or not member.name.startswith(archive_prefix):
                    continue
                relative = Path(member.name.removeprefix(archive_prefix))
                if (
                    len(relative.parts) == 2
                    and relative.parts[0] in EXPECTED_COUNTS
                    and TASK_NAME_PATTERN.fullmatch(relative.parts[1]) is not None
                ):
                    selected.append((member, relative))

            selected_counts = {
                split: sum(relative.parts[0] == split for _, relative in selected)
                for split in EXPECTED_COUNTS
            }
            if selected_counts != EXPECTED_COUNTS:
                raise RuntimeError(
                    f'Archive task counts differ: {selected_counts}'
                )

            for member, relative in selected:
                destination = staging_root / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                source = archive.extractfile(member)
                if source is None:
                    raise RuntimeError(f'Cannot read archive member: {member.name}')
                with source, destination.open('wb') as output:
                    while chunk := source.read(1024 * 1024):
                        output.write(chunk)

    file_hashes, counts = build_file_manifest(staging_root)
    dataset_bytes = sum(path.stat().st_size for path in staging_root.rglob('*.json'))
    manifest = {
        'schema_version': 1,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'source': {
            'repository': SOURCE_REPOSITORY,
            'commit': SOURCE_COMMIT,
            'archive_sha256': EXPECTED_ARCHIVE_SHA256,
        },
        'path': str(DATASET_ROOT),
        'counts': counts,
        'json_bytes': dataset_bytes,
        'files': file_hashes,
    }

    DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    staging_root.replace(DATASET_ROOT)
    atomic_write_json(MANIFEST_PATH, manifest)
    manifest = verify_installed_dataset()
    print('ARC-AGI-2 installed and verified.')

{
    'path': manifest['path'],
    'counts': manifest['counts'],
    'json_mib': manifest['json_bytes'] / 2**20,
    'manifest': str(MANIFEST_PATH),
}


## Completion

The overfitting notebook expects both the dataset directory and manifest above.
The raw data stays outside Git and can be reused by later Modal experiments.
